# Social Network Visualisation

**Input:** Telegram messages data, raw from data provider (e.g. pandas dataframes or csv/xlsx/json files)  

**Dependencies:** None

In [70]:
import pandas as pd
import numpy as np
from collections import Counter
import networkx as nx

In [71]:
# Read data

df = pd.read_csv("messages_latin_10K.csv")
df = df[['date', 'forwards', 'nr_replies', 'id', 'views', 'message', 'reactions', 'channel_id', 'fwd_from_id', 'from_id']]

channels = pd.read_csv('df_latin.csv')[['about', 'id',
                   'participants_count', 'message_count', 'title', 'verified', 'username', 'query_id']]
channels = channels.drop_duplicates(subset=['id'])

/tmp/ipykernel_116/3886260528.py:6: DtypeWarning: Columns (83,87,88,89,91,92,93,94,96) have mixed types. Specify dtype option on import or set low_memory=False.
  channels = pd.read_csv('df_latin.csv')[['about', 'id',


In [72]:
print(df.shape, channels.shape)

(3871, 10) (10980, 8)


## Import channel usernames

In [73]:
if len( set(df.channel_id) & set(channels.id) ) == df.channel_id.nunique():
    print("All channels IDs in the current messages matched with the original channel metadata files")
else:
    print("WARNING: Some channels in the current messages were NOT matched with the original channel metadata files")

All channels IDs in the current messages matched with the original channel metadata files


In [74]:
# Rename id column
channels = channels.rename(columns={'id': 'channel_id'})

# Merge datasets
df = df.merge(channels, how='left', on='channel_id')
df.shape

(3871, 17)

In [75]:
print(f"Numer of missing usernames: { df[df['username'].isna()].channel_id.nunique() }")
print(f"Numer of missing titles: { df[df['title'].isna()].channel_id.nunique() }")

Numer of missing usernames: 26
Numer of missing titles: 0


## Build network

In [76]:
# Filter on messages forwarded from authors in the dataset
df = df[(df.fwd_from_id.isin(channels['channel_id'])) & (df.fwd_from_id.notna())]
print("Number of forwarded messages in this data:", len(df))

Number of forwarded messages in this data: 452


In [77]:
df[['channel_id','title','fwd_from_id']].drop_duplicates()

,channel_id,title,fwd_from_id
165,1805758411,Albatrops,1.245676e+09
180,1540805607,Nuevos paradigmas,1.750482e+09
181,1540805607,Nuevos paradigmas,1.580590e+09
248,1702605423,RedVakulinchuk Chat,1.251137e+09
263,2026839224,Overton-Magazin Kommentare,1.703058e+09
...,...,...,...
3725,1915099177,Bayern Steht Zusammen - Talk,1.240327e+09
3727,1915099177,Bayern Steht Zusammen - Talk,1.486170e+09
3733,2161509130,IN DIE GENE Chat,1.520416e+09
3734,1512222527,❤️ GESUNDHEIT & FREI VON MEDIKAMENTEN ❤️,1.544679e+09


In [78]:
# Create edge list

edges = df[['fwd_from_id','channel_id']]
print(f"Total number of directed edges: {len(edges)}")

edges = edges.groupby(['fwd_from_id', 'channel_id']).size().reset_index(name='weight')
print(f"Total number of unique, weighted directed edges: {len(edges)}")
edges.sample(5)

Total number of directed edges: 452
Total number of unique, weighted directed edges: 76


,fwd_from_id,channel_id,weight
9,1.191903e+09,1057004609,1
17,1.251137e+09,1702605423,2
45,1.520416e+09,2161509130,1
73,2.110727e+09,1057004609,3
55,1.592366e+09,1452083659,1


In [79]:
# Build network

# Assuming df_weighted has columns: ['source', 'target', 'weight']
G = nx.DiGraph()

# Add edges with weight
for _, row in edges.iterrows():
    G.add_edge(row['fwd_from_id'], row['channel_id'], weight=row['weight'])

# ✅ G is now a directed weighted graph
print(G)

DiGraph with 106 nodes and 76 edges


In [80]:
# Add channel title as node label attribute
id_to_title = channels.set_index('channel_id')['title'].to_dict()
nx.set_node_attributes(G, id_to_title, name='label')

# Add weights as edge attributes
in_deg_w = dict(G.in_degree(weight='weight'))  # weighted
in_deg_w = {k: float(v) for k, v in in_deg_w.items()}
out_deg_w = dict(G.out_degree(weight='weight'))  # weighted
tot_deg_w = dict(G.degree(weight='weight'))  # weighted

nx.set_node_attributes(G, in_deg_w, name='in_degree_w')
nx.set_node_attributes(G, out_deg_w, name='out_degree_w')
nx.set_node_attributes(G, tot_deg_w, name='total_degree_w')

# # Relabel
# mapping = id_to_title
# G = nx.relabel_nodes(G, mapping)

# Check
list(G.nodes(data=True))[:5]

[(1081286887.0,
  {'label': 'Product Science by Anton Martsen',
   'in_degree_w': 0.0,
   'out_degree_w': 12.0,
   'total_degree_w': 12.0}),
 (1350800797.0,
  {'label': 'Product Scientists',
   'in_degree_w': 12.0,
   'out_degree_w': 0,
   'total_degree_w': 12.0}),
 (1085541350.0,
  {'label': 'sam omi',
   'in_degree_w': 0.0,
   'out_degree_w': 1.0,
   'total_degree_w': 1.0}),
 (1820731265.0,
  {'label': 'norulers',
   'in_degree_w': 1.0,
   'out_degree_w': 0,
   'total_degree_w': 1.0}),
 (1100528049.0,
  {'label': 'Mad in Germany ©️',
   'in_degree_w': 0.0,
   'out_degree_w': 1.0,
   'total_degree_w': 1.0})]

In [81]:
# ✅ 1. Check if any node is missing the label attribute
missing_label_nodes = [n for n, attrs in G.nodes(data=True) 
                       if 'label' not in attrs]

len(missing_label_nodes), missing_label_nodes[:10]

(0, [])

In [82]:
# ✅ 2. Check if any node has a label attribute but it is NA / None
na_label_nodes = [n for n, attrs in G.nodes(data=True) 
                  if attrs.get('label') is None or pd.isna(attrs.get('label'))]

len(na_label_nodes), na_label_nodes[:10]

(0, [])

## Meta-data attribution (example: Louvain cluster label)

In [83]:
# !pip install python-louvain

In [84]:
# # Detect Louvain communities

# from community import community_louvain # https://github.com/taynaud/python-louvain
# partition = community_louvain.best_partition(G.to_undirected()) # {node: community ID}

In [85]:
# nx.set_node_attributes(G, partition, 'community')

In [86]:
# print(max(partition.values()))

## Export as gexf file

In [87]:
# Export to gexf for GephiLite viz

nx.write_gexf(G, "graph.gexf")

Instructions for GephiLite visualisation:  
- layout: ForceAtlas2
- node color: community (is node attribute)
- node size: degree (computed by Gephi)
- edge size: as close to zero as possible OR weight (is edge attribute)
- labels: no labels

**Ouput**: gexf file

## Export filtered df

In [88]:
df.to_csv('network_df.csv', index=False)